In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = sns.load_dataset("titanic")

In [ ]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


Statistics for all columns

In [ ]:
df.describe(include='all')

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
count,891.000000,891.000000,891,714.000000,891.000000,891.000000,891.000000,889,891,891,891,203,889,891,891
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,3,3,3,2,7,3,2,2
top,NaN,NaN,male,NaN,NaN,NaN,NaN,S,Third,man,True,C,Southampton,no,True
freq,NaN,NaN,577,NaN,NaN,NaN,NaN,644,491,537,537,59,644,549,537
mean,0.383838,2.308642,NaN,29.699118,0.523008,0.381594,32.204208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.486592,0.836071,NaN,14.526497,1.102743,0.806057,49.693429,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,1.000000,NaN,0.420000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,0.000000,2.000000,NaN,20.125000,0.000000,0.000000,7.910400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,0.000000,3.000000,NaN,28.000000,0.000000,0.000000,14.454200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.000000,3.000000,NaN,38.000000,1.000000,0.000000,31.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


A column indicating the number of relatives for each passenger on board, instead of:
number of brothers/sisters or husbands/wives on board;
number of parents or children on board

In [ ]:
df['relatives'] = df['sibsp']+df['parch']
check_alone=df[df['alone']==True]['relatives'].unique()
df.drop(columns=['sibsp', 'parch', 'alone'], inplace=True)
nan_count=int(df['relatives'].isna().sum())
nan_count


0

How often a certain number of relatives appear in a new column

In [ ]:
stat_relatives=df.groupby('relatives')['relatives'].agg('count')
stat_relatives

,relatives
relatives,
0,537
1,161
2,102
3,29
4,15
5,22
6,12
7,6
10,7


Replace the number of relatives exceeding 5(five) with the value "above 5"(over five)

In [ ]:
categories=[]
for value in df['relatives']:
  if value>5:
    categories.append('above 5')
  else:
    categories.append(str(value))
df['relatives_category']=categories
stat_relatives_categ=df.groupby('relatives_category')['relatives_category'].agg('count')
stat_relatives_categ

,relatives_category
relatives_category,
0,537
1,161
2,102
3,29
4,15
5,22
above 5,25


Statistics on the percentage of passengers with more than 5 relatives relative to the total number of passengers for each of the cities of departure

In [ ]:
stat_city=df.groupby('embark_town')['embark_town'].agg(['count'])
stat_city.columns=['total']
stat_city['above5_onl']=df[df['relatives_category']=='above 5'].groupby('embark_town')['embark_town'].agg('count')
stat_city['above5_onl']=stat_city['above5_onl'].fillna(0)
stat_city['perc']=np.round(stat_city['above5_onl']/stat_city['total']*100).astype(int)
stat_city

,total,above5_onl,perc
embark_town,,,
Cherbourg,168,0.0,0
Queenstown,77,0.0,0
Southampton,644,25.0,4


Filling in missing age values ​​with the median

In [ ]:
age_median=df['age'].median()
df.loc[df['age'].isna(), 'age']=age_median
age_median

28.0

Create a new column where age is represented by a category instead of a number

In [ ]:
def classify_age(age):
  if pd.isna(age):
    return 'Unknown'
  elif age<14:
    return 'Up to 14'
  elif 14<=age<=34:
    return '14-34 years old'
  elif 35<=age<=59:
    return '35-59 years old'
  else:
    return '60 years and older'
df['age_group'] = df['age'].apply(classify_age)
stat_age=df.groupby('age_group')['age_group'].agg('count')
stat_age

,age_group
age_group,
14-34 years old,407
35-59 years old,209
60 years and older,27
Unknown,177
Up to 14,71


Which age group had the highest mortality rate?

In [ ]:
df['is_alive']=df['alive']=='yes'
total_by_age = df.groupby('age_group')['age_group'].agg('count')
died_by_age=df[df['is_alive']==False].groupby('age_group')['age_group'].agg('count')
mr_rate=(died_by_age/total_by_age*100).round(2)
mr_rate

,age_group
age_group,
14-34 years old,62.16
35-59 years old,58.37
60 years and older,74.07
Unknown,70.62
Up to 14,40.85


Detailed mortality statistics by age category, ticket class, cabin level and number of relatives

In [ ]:
age_mr=(1-df.groupby('age_group')['survived'].mean())*100
class_mr=(1-df.groupby('pclass')['survived'].mean())*100
deck_mr=(1-df.groupby('deck')['survived'].mean())*100
relat_mr=(1-df.groupby('relatives_category')['survived'].mean())*100
print(age_mr, class_mr, deck_mr, relat_mr)

age_group
14-34 роки           64.726027
35-59 років          58.373206
60 і більше років    74.074074
До 14                40.845070
Name: survived, dtype: float64 pclass
1    37.037037
2    52.717391
3    75.763747
Name: survived, dtype: float64 deck
A    53.333333
B    25.531915
C    40.677966
D    24.242424
E    25.000000
F    38.461538
G    50.000000
Name: survived, dtype: float64 relatives_category
0          69.646182
1          44.720497
2          42.156863
3          27.586207
4          80.000000
5          86.363636
above 5    84.000000
Name: survived, dtype: float64


/tmp/ipykernel_11451/3094042414.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  deck_mr=(1-df.groupby('deck')['survived'].mean())*100


In the age data, the highest mortality rate is observed among people in the age group 60 and over (74.07%). It can be assumed that the reason is the difficulty of evacuation for elderly passengers. In the ticket class data, the highest mortality rate is observed among people in class 3 (75.8%). Perhaps the reason is socio-economic status. In the cabin level data, we see that the highest mortality rate is in cabins of class A (53.3%). Perhaps the distance from these cabins to the weddings was the longest. In the data on relatives, it is clear that the highest mortality rate is observed in people who were with 5 relatives (86.36%). Therefore, the highest mortality rate is observed in people in the age group 60 and older, 3rd class of ticket, with 5 relatives and cabin level A. In my opinion, the greatest influence on mortality is the class of ticket, since the data on classes has the greatest discrepancy. But the number of relatives is also an important factor.